<a href="https://colab.research.google.com/github/Aparimita18/Question_Answering_Model_on_NLP/blob/main/Question_Answering_Model_on_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install PyMuPDF transformers gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.6 MB/s eta 0:00:00


PDF extraction

In [2]:
import fitz

book = fitz.open("/content/dataset.pdf")

# Extract text from all pages
data = ""
for page in book:
    data += page.get_text()

print(data[:1000])

Speech and Language Processing
An Introduction to Natural Language Processing,
Computational Linguistics, and Speech Recognition
with Language Models
Third Edition draft
Daniel Jurafsky
Stanford University
James H. Martin
University of Colorado at Boulder
Copyright ©2024. All rights reserved.
Draft of January 12, 2025. Comments and typos welcome!
Summary of Contents
I
Fundamental Algorithms for NLP
1
1
Introduction. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
3
2
Regular Expressions, Tokenization, Edit Distance . . . . . . . . . . . . . . .
4
3
N-gram Language Models . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
32
4
Naive Bayes, Text Classiﬁcation, and Sentiment . . . . . . . . . . . . . . . . . 56
5
Logistic Regression . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 77
6
Vector Semantics and Embeddings . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10

In [3]:
import fitz  # PyMuPDF
import pandas as pd

# Open PDF
doc = fitz.open("dataset.pdf")
page = doc[0]  # First page

# returns list of dicts with positional info
blocks = page.get_text("dict")["blocks"]

# Filter for text blocks
table_data = []
for b in blocks:
    if b["type"] == 0:  # text
        for line in b["lines"]:
            row = []
            for span in line["spans"]:
                row.append(span["text"].strip())
            if row:
                table_data.append(row)

# convert to DataFrame
for row in table_data:
    print(row)

df = pd.DataFrame(table_data)
print(df)


['Speech and Language Processing']
['An Introduction to Natural Language Processing,']
['Computational Linguistics, and Speech Recognition']
['with Language Models']
['Third Edition draft']
['Daniel Jurafsky']
['Stanford University']
['James H. Martin']
['University of Colorado at Boulder']
['Copyright ©2024. All rights reserved.']
['Draft of January 12, 2025. Comments and typos welcome!']
                                                    0
0                      Speech and Language Processing
1     An Introduction to Natural Language Processing,
2   Computational Linguistics, and Speech Recognition
3                                with Language Models
4                                 Third Edition draft
5                                     Daniel Jurafsky
6                                 Stanford University
7                                     James H. Martin
8                   University of Colorado at Boulder
9               Copyright ©2024. All rights reserved.
10  Draft of 

Text Cleaning

In [4]:
import re

def clean(text,debug= True):

    text = re.sub(r"-\n", "", text)
    # Replace multiple newlines with a single one
    text = re.sub(r"\n{2,}", "\n\n", text)


    text = re.sub(r'(Fig(?:ure)?\.? ?\d+(\.\d+)?(?:[^\w\n]{0,20})?)', '', text)
    text = re.sub(r"\(?[Tt]he interested reader.*?Section \d+(\.\d+)*.*?\)?", "", text)
    text = re.sub(r"\(?see (Section|Chapter|Subsection) \d+(\.\d+)*.*?\)?", "", text)
    text = re.sub(r"\(?refer to (Section|Chapter|Subsection) \d+(\.\d+)*.*?\)?", "", text)
    text = re.sub(r"\(?(as|as we) (see|saw) (in )?(Section|Chapter|Subsection) \d+(\.\d+)*.*?\)?", "", text)
    text = re.sub(r'^\s*[\*\-•]+.*$', '', text, flags=re.MULTILINE)
    #paragraphs = re.split(r'\n?(?=\d+(\.\d+)*\s+[^\n]+)', text)
    text = re.sub(r'^\s*\d+(\.\d+)*\s+[•\-–—]\s+.+?\d\s*$', '', text, flags=re.MULTILINE)




    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()

        # Regular expressions to handle various cases
        if re.match(r'^\d+\s+Chapter\s+\d+', line, re.IGNORECASE):
            continue
        if re.match(r'^Chapter\s+\d+', line, re.IGNORECASE):
            continue
        if re.fullmatch(r'\d+\s*[•\-–—]?', line) or re.fullmatch(r'[•\-–—]?\s*\d+', line):
            continue
        if re.fullmatch(r'\d+', line):
            continue
        if re.fullmatch(r'\d*\.\d+', line):
            continue
        if re.fullmatch(r'(Figure|Table|Section|Subsection)\s+\d+(\.\d+)*', line, re.IGNORECASE):
            continue
        if re.fullmatch(r'^•\s.*\s[\d.]+$',line):
            continue
        if re.search(r"(Speech and Language Processing|Jurafsky|Martin|Draft of \d{4})", line, re.IGNORECASE):
            continue
        if re.match(r'^[>\-•*]?\s*[A-Z\s]+$', line):
            continue
        if re.fullmatch(r'[\d\s•\-–—]*', line):
            continue
        if re.search(r"(C HAPTER|Speech and Language Processing|Jurafsky|Martin|Draft of \d{4}|CHAPTER \d+|N- GRAM|SPEECH RECOGNITION)", line, re.IGNORECASE):
            continue
        if re.fullmatch(r'[•\-–—]\s+[A-Z\s]{3,}', line) or line.isupper():
            continue
        if re.search(r'(Figure|Fig\.|Table|Box|Image|Diagram)\s*\d+(\.\d+)?', line, re.IGNORECASE):
            continue
        if re.fullmatch(r'[\[\(]?(image|diagram|fig|figure|illustration)[\]\)]?', line, re.IGNORECASE):
            continue
        if len(re.findall(r'[A-Za-z]', line)) < 4:
            continue
        if re.fullmatch(r'[•\-–—]\s+[A-Z\s]', line):
            continue

        cleaned_lines.append(line)

    cleaned_text = "\n".join(cleaned_lines)

    return cleaned_text


In [5]:
def splitpara(text):
    paragraphs = []
    current = ""

# Define known footer patterns (can extend this as needed)
    footer_patterns = [
        r'^For example if we use both begin.*',
        r'^Speech and Language Processing.*',
        r'^Jurafsky.*',
        r'^Draft of \d{4}.*',
        r'^Page \d+.*',
        r'^Figure \d+.*',
        r'^.*\b\d{3,}\b.*$'  # Lines with page numbers
    ]

    # Clean up footers
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        if any(re.match(pat, line.strip()) for pat in footer_patterns):
            continue
        cleaned_lines.append(line)

    # Join back into cleaned text
    cleaned_text = '\n'.join(cleaned_lines)

    # paragraph splitting based on subsection heading and punctuation
    lines = cleaned_text.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Remove figure/image/table mentions
        line = re.sub(r'(Fig(?:ure)?\.?\s*\d+(\.\d+)?(?:[^\w\n]{0,20})?)', '', line)
        line = re.sub(r'(Table\s*\d+(\.\d+)?(?:[^\w\n]{0,20})?)', '', line)
        line = re.sub(r'(Box\s*\d+(\.\d+)?(?:[^\w\n]{0,20})?)', '', line)
        line = re.sub(r'\(?(see|refer to)?\s*(Fig(?:ure)?|Table|Box)\s*\d+(\.\d+)?\)?', '', line, flags=re.IGNORECASE)


        if re.match(r'^\d+(\.\d+)+\s+[A-Za-z]', line):
            if current.strip():
                paragraphs.append(current.strip())
                current = ""
            paragraphs.append(line.strip())
            continue

        # If line ends with a period
        if line.endswith('.'):
            current += " " + line
            paragraphs.append(current.strip())
            current = ""
        else:
            current += " " + line

    # Add remaining paragraph
        if current.strip():
          paragraphs.append(current.strip())

    #splitting on whitespace i.e 2 or more newlines
    final_paragraphs = []
    for para in paragraphs:
        chunks = re.split(r'\n\s*\n', para)
        final_paragraphs.extend([chunk.strip() for chunk in chunks if chunk.strip()])

    return final_paragraphs
    #return paragraphs

In [6]:
cleaned_text = clean(data)
paragraphs = splitpara(cleaned_text)

import textwrap
print(f"Total paragraphs: {len(paragraphs)}")
print(textwrap.fill(paragraphs[105]))

Total paragraphs: 25142
Exercises Vector Semantics and Embeddings Vector Semantics TF-IDF:
Weighing terms in the vector Word2vec Exercises Neural Networks
Training Neural Nets RNNs and LSTMs Summary: Common RNN NLP
Architectures The Transformer Parallelizing computation using a single
matrix X The input: embeddings for token and position The Language
Modeling Head 10 Large Language Models Sampling for LLM Generation
Pretraining Large Language Models Potential Harms from Language Models
11 Masked Language Models Fine-Tuning for Sequence Labelling: Named
Entity Recognition 12 Model Alignment, Prompting, and In-Context
Learning Model Alignment with Human Preferences: RLHF and DPO NLP
Applications 13 Machine Translation Bias and Ethical Issues Exercises
14 Question Answering, Information Retrieval, and RAG Information
Retrieval Answering Questions with RAG Exercises 15 Chatbots &
Dialogue Systems Properties of Human Conversation Exercises Feature
Extraction for ASR: Log Mel Spectrum ASR Ev

In [7]:
import pandas as pd

df = pd.DataFrame(paragraphs, columns=["text"])
df.to_csv("dataset.csv", index=False,sep="|", encoding="utf-8", header=True)
print("Saved as dataset.csv")

from google.colab import files
#files.download("dataset.csv")

Saved as dataset.csv


In [8]:


print(textwrap.fill(df['text'][1839]))

Smoothing, Interpolation, and Backoff There is a problem with using
maximum likelihood estimates for probabilities: any ﬁnite training
corpus will be missing some perfectly acceptable English word
sequences. That is, cases where a particular n-gram never occurs in
the training data but appears in the test set. Perhaps our training
corpus has the words ruby and slippers in it but just happens not to
have the phrase ruby slippers.


## **Re-check if the data extraction & cleaning is done properly**

In [9]:

#list of all paragraphs
paragraphs = df['text']
print(paragraphs)
print("\n")
print(len(paragraphs))

0          An Introduction to Natural Language Processing,
1        An Introduction to Natural Language Processing...
2        An Introduction to Natural Language Processing...
3        An Introduction to Natural Language Processing...
4        An Introduction to Natural Language Processing...
                               ...                        
25137    fragment of word, 13 Gaussian prior on weights...
25138    fragment of word, 13 Gaussian prior on weights...
25139    fragment of word, 13 Gaussian prior on weights...
25140    fragment of word, 13 Gaussian prior on weights...
25141    fragment of word, 13 Gaussian prior on weights...
Name: text, Length: 25142, dtype: object


25142


In [10]:


print(f"Total pages in PDF: {len(book)}")

Total pages in PDF: 599


In [11]:

print(f"Total paragraphs extracted: {len(paragraphs)}")

Total paragraphs extracted: 25142


In [12]:
print(textwrap.fill(df['text'][12530]))

These changes in air pressure obviously originate with the speaker and
are caused by the speciﬁc way that air passes through the glottis and
out the oral or nasal cavities. We represent sound waves by plotting
the change in air pressure over time.


In [13]:
import random

for _ in range(5):
    parano = random.randint(0, len(paragraphs)-1)
    print(f"--- Paragraph {parano} ---")
    print(textwrap.fill(paragraphs[parano]))
    print("\n\n")

--- Paragraph 5530 ---
More details on feedforward networks Let’s now set up some notation to
make it easier to talk about deeper networks of



--- Paragraph 13062 ---
The alternative model is to use encoder-decoder models, which have
been shown to work better than rules for such transduction tasks, but
do require expert-labeled training sets in which non-standard words
have been replaced with the appropriate In the simplest encoder-
decoder setting, we simply treat the problem like machine translation,
training a system to map from: They live at two twenty four Mission
Street While encoder-decoder algorithms are highly accurate, they
occasionally produce errors that are egregious; for example
normalizing 45 minutes as forty ﬁve millimeters. To address this, more
complex systems use mechanisms like lightweight covering grammars,
which enumerate a large set of possible verbalizations but



--- Paragraph 18732 ---
Lexicon-based methods for Entity-Centric Affect What if we want to get
a

In [14]:
chapters = [
  "Introduction",
  "Regular Expressions, Tokenization, Edit Distance",
  "N-gram Language Models",
  "Naive Bayes, Text Classification, and Sentiment",
  "Logistic Regression",
  "Vector Semantics and Embeddings",
  "Neural Networks",
  "RNNs and LSTMs",
  "The Transformer",
  "Large Language Models",
  "Masked Language Models",
  "Model Alignment, Prompting, and In-Context Learning",
  "Machine Translation",
  "Question Answering, Information Retrieval, and RAG",
  "Chatbots & Dialogue Systems",
  "Automatic Speech Recognition and Text-to-Speech",
  "Annotating Linguistic Structure",
  "Sequence Labeling for Parts of Speech and Named Entities",
  "Context-Free Grammars and Constituency Parsing",
  "Dependency Parsing",
  "Information Extraction: Relations, Events, and Time",
  "Semantic Role Labeling",
  "Lexicons for Sentiment, Affect, and Connotation",
  "Coreference Resolution and Entity Linking",
  "Discourse Coherence",
]

In [15]:
!pip install fuzzywuzzy


In [16]:
from fuzzywuzzy import fuzz



/usr/local/lib/python3.11/dist-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [17]:

matches = {}
for section in chapters :
    found = any(fuzz.partial_ratio(section.lower(), para.lower()) > 85 for para in paragraphs)

    #found = any(section.lower() in para.lower() for para in paragraphs)
    matches[section] = "✅ Found" if found else "❌ Not Found"

for k, v in matches.items():
    print(f"{k}: {v}")

Introduction: ✅ Found
Regular Expressions, Tokenization, Edit Distance: ✅ Found
N-gram Language Models: ✅ Found
Naive Bayes, Text Classification, and Sentiment: ✅ Found
Logistic Regression: ✅ Found
Vector Semantics and Embeddings: ✅ Found
Neural Networks: ✅ Found
RNNs and LSTMs: ✅ Found
The Transformer: ✅ Found
Large Language Models: ✅ Found
Masked Language Models: ✅ Found
Model Alignment, Prompting, and In-Context Learning: ✅ Found
Machine Translation: ✅ Found
Question Answering, Information Retrieval, and RAG: ✅ Found
Chatbots & Dialogue Systems: ✅ Found
Automatic Speech Recognition and Text-to-Speech: ✅ Found
Annotating Linguistic Structure: ✅ Found
Sequence Labeling for Parts of Speech and Named Entities: ✅ Found
Context-Free Grammars and Constituency Parsing: ✅ Found
Dependency Parsing: ✅ Found
Information Extraction: Relations, Events, and Time: ✅ Found
Semantic Role Labeling: ✅ Found
Lexicons for Sentiment, Affect, and Connotation: ✅ Found
Coreference Resolution and Entity Linki

##**Chunk text into tokens for BERT**

In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
def chunk_paragraphs(paragraphs, tokenizer, max_length=512, stride=64):
    """
    Tokenizes and chunks paragraphs into BERT-compatible segments using a sliding window.
    Returns a list of dictionaries with input_ids, attention_mask, and original text.
    """
    chunked = []

    for idx, para in enumerate(paragraphs):
        encodings = tokenizer(
            para,
            return_overflowing_tokens=True,
            truncation=True,
            max_length=max_length,
            stride=stride,
            return_attention_mask=True,
            return_offsets_mapping=True,
            padding=False
        )

        for i in range(len(encodings["input_ids"])):
            chunk_text = tokenizer.decode(encodings["input_ids"][i], skip_special_tokens=True)
            chunked.append({
                "para_index": idx,
                "chunk_index": i,
                "input_ids": encodings["input_ids"][i],
                "attention_mask": encodings["attention_mask"][i],
                "text": chunk_text
            })

    return chunked


In [ ]:
# Use your previously prepared paragraphs
paragraphs = df['text'].tolist()  # assuming df from earlier

# Tokenize and chunk
chunked_data = chunk_paragraphs(paragraphs, tokenizer)

# Example output
print(f"Total chunks: {len(chunked_data)}")
print("Sample chunk text:\n", chunked_data[10001]["text"])


Generate manual QA pairs

In [ ]:
import pandas as pd

# Load paragraph dataset
paragraphs = pd.read_csv('/content/dataset.csv', sep = '|')
paragraphs = paragraphs.dropna().reset_index(drop=True)
#paragraphs.head()


In [ ]:
'''# Create an empty DataFrame for manual QA
manual_qa = pd.DataFrame(columns=['question', 'answer'])
#example
new_row = pd.DataFrame([{'question': 'What is instruction tuning?',
                               'answer': 'Instruction tuning (short for instruction finetuning, and sometimes even shortened to instruct tuning) is a method for making an LLM better at following instructions. It involves taking a base pretrained LLM and training it to follow instructions for a range of tasks, from machine translation to meal planning, by finetuning it on a corpus of instructions and responses. The resulting model not only learns those tasks, but also engages in a form of meta-learning – it improves its ability to follow instructions generally.Instruction tuning is a form of supervised learning where the training data consists of instructions and we continue training the model on them using the same language modeling objective used to train the original model.'
                               }])

manual_qa = pd.concat([manual_qa, new_row], ignore_index=True)

# Save once done
#manual_qa.to_csv('manual.csv', index=False)
'''

In [ ]:
import pandas as pd
data = pd.read_csv('/content/manual.csv',sep = ',')
data

Generate automatic QA pairs

In [ ]:

!pip install sentencepiece

from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch
import pandas as pd
from tqdm import tqdm
import nltk
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt')

In [ ]:
# Load the pre-trained T5 model and tokenizer
model = T5ForConditionalGeneration.from_pretrained("valhalla/t5-base-qg-hl")
tokenizer = T5Tokenizer.from_pretrained("valhalla/t5-base-qg-hl")


In [ ]:
#Load clean dataset
df = pd.read_csv("dataset.csv", sep='|')
df.dropna(subset=['text'], inplace=True)
df.reset_index(drop=True, inplace=True)

# function to generate question
def generate_question(context, answer):
    input_text = f"generate question: {context} </s>"
    input_ids = tokenizer.encode(input_text, return_tensors="pt", max_length=512, truncation=True)

    outputs = model.generate(
        input_ids=input_ids,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question



# store generated QA pairs
auto = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    paragraph = row['text']
    sentences = sent_tokenize(paragraph)
    if len(sentences) == 0:
        continue
    answer = sentences[0]  # using the first sentence as answer for now
    context = paragraph.replace(answer, f"<hl> {answer} <hl>")

    try:
        question = generate_question(context, answer)
        auto.append({"question": question, "answer": answer})
    except:
        continue  # skip if error in generation

# Create DataFrame from auto-generated QA pairs
auto_df = pd.DataFrame(auto)


auto_df.to_csv("automatic.csv", index=False)
print("✅ Automatic QA generation completed and saved.")


Streaming output truncated to the last 5000 lines.
 52%|█████▏    | 13008/25142 [8:48:15<14:30:38,  4.31s/it]